In [2]:
# Install pyspark and findspark
!pip install pyspark findspark pandas openpyxl

from google.colab import drive
drive.mount('/content/drive')

import findspark
findspark.init()  # Initialize Spark

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd

# Initialize Spark session
spark = SparkSession.builder \
    .appName("LinearRegressionExample") \
    .getOrCreate()

# Read Excel file using pandas (PySpark doesn't natively read Excel)
# Save your Excel file as 'data.xlsx' in your working directory
pandas_df = pd.read_excel('/content/drive/MyDrive/BigBoy.xlsx')
spark_df = spark.createDataFrame(pandas_df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Assume your features are 'feature1', 'feature2', ..., and target is 'target'
from pyspark.sql.functions import unix_timestamp, col
from datetime import datetime

# Convert 'Date' column to Unix timestamp (numeric) before assembling features
spark_df_numeric_date = spark_df.withColumn('Date', unix_timestamp(spark_df['Date']))

feature_cols = ['Date', 'Avg Predicted CPI', 'Market Confidence Score', 'Trading Intensity', 'Fed Interest Rate (%)', 'Unemployment Rate (%)', 'Oil Prices (USD/bbl)', 'Housing Prices', 'S&P 500']  # Replace with your actual feature names

# Drop rows with any NaN values from the relevant columns (features and label) to prevent VectorAssembler errors
all_relevant_cols = feature_cols + ['CPI']
cleaned_df = spark_df_numeric_date.dropna(subset=all_relevant_cols)

# Define the date range in Unix timestamps
start_date = int(datetime(2022, 11, 1).timestamp())
end_date = int(datetime(2025, 12, 31, 23, 59, 59).timestamp())

# Filter the DataFrame based on the date range
cleaned_df = cleaned_df.filter(col('Date').between(start_date, end_date))

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(cleaned_df)

In [4]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

In [5]:
# Initialize and fit the model
lr = LinearRegression(featuresCol="features", labelCol="CPI")
model = lr.fit(train_data)

In [7]:
# Make predictions on the test data
predictions = model.transform(test_data)

# Evaluate the model using RegressionEvaluator
evaluator = RegressionEvaluator(labelCol="CPI", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on test data = {rmse}")

evaluator_r2 = RegressionEvaluator(labelCol="CPI", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)
print(f"R-squared (R2) on test data = {r2}")

Root Mean Squared Error (RMSE) on test data = 1.538157730898566
R-squared (R2) on test data = 0.9468792894154006


In [9]:
import pandas as pd

# Get feature names from the assembler
feature_names = assembler.getInputCols()

# Get coefficients and intercept
coefficients = model.coefficients
intercept = model.intercept

# Create a Pandas DataFrame for better visualization
coefficient_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients.toArray()
})

print("Feature Coefficients:")
display(coefficient_df.sort_values(by='Coefficient', ascending=False))
print(f"\nIntercept: {intercept}")

Feature Coefficients:


,Feature,Coefficient
1,Avg Predicted CPI,9.732592e-02
4,Fed Interest Rate (%),7.671648e-02
6,Oil Prices (USD/bbl),6.312452e-02
2,Market Confidence Score,1.017324e-02
0,Date,3.238360e-07
3,Trading Intensity,-3.737428e-06
7,Housing Prices,-1.861869e-05
8,S&P 500,-1.709977e-04
5,Unemployment Rate (%),-2.583682e+00



Intercept: -229.8289934294569


In [10]:
predictions_df = predictions.select('CPI', 'prediction').toPandas()

# Save the pandas DataFrame to an Excel file
output_excel_path = '/content/drive/MyDrive/LinearRegressionOutput.xlsx'
predictions_df.to_excel(output_excel_path, index=False)

print(f"Predictions saved to {output_excel_path}")

Predictions saved to /content/drive/MyDrive/LinearRegressionOutput.xlsx
